# 🍅 Tomato-Oversight — Colab 부트스트랩

**2026 창의설계축전** 팀 공용 시작 노트북.
이 노트북을 위에서부터 순서대로 실행하면 재배 로봇 환경이 Colab에서 세팅되고, 격자를 시각화할 수 있습니다.

- **처음 오는 팀원**: `런타임 > 모두 실행`을 눌러 환경이 정상 동작하는지 확인하세요.
- GPU는 필요 없습니다 (`런타임 > 런타임 유형 변경`을 **CPU**로 둬도 충분합니다).
- 학습 코드는 아직 없습니다 — 이 노트북은 "환경이 도는지" 확인하는 용도입니다.
- 실제 학습은 이 노트북을 복사해 `honest_train.ipynb`처럼 각자 만들어 진행하세요.


## 1. 설정 — 패키지 설치 + 팀 코드 내려받기

아래 셀은 매 세션 한 번 실행합니다 (Colab은 세션이 끊기면 설치가 사라집니다).

> ⚠️ 저장소가 **Private**이면 `git clone`이 실패합니다. 그 경우 저장소를 Public으로 바꾸거나, Personal Access Token이 필요합니다 (담당자에게 문의).


In [ ]:
!pip install -q stable-baselines3 gymnasium

import os, sys

# Colab이면 /content 아래에 저장소를 clone하고 그 폴더로 이동합니다.
if os.path.isdir('/content'):
    os.chdir('/content')
    if not os.path.isdir('tomato-oversight'):
        !git clone https://github.com/1ee1ee1ee/tomato-oversight.git
    os.chdir('/content/tomato-oversight')

# 저장소 루트를 import 경로에 추가 (from src.grower_env ... 가 되도록)
sys.path.insert(0, os.getcwd())
print('working dir :', os.getcwd())
print('src 존재    :', os.path.isdir('src'))


## 2. 환경 불러오기 & 동작 확인

`TomatoWateringEnv`를 만들고 리셋한 뒤, 텍스트(ansi)로 격자를 출력합니다.
- `R` = 재배 로봇, `O` = 스푸핑 타일, `T` = 살아있는 토마토, `x` = 죽은 토마토, `.` = 빈 칸


In [ ]:
from src.grower_env import TomatoWateringEnv, GrowerConfig, Action

env = TomatoWateringEnv(GrowerConfig(mode='E1'))
obs, info = env.reset(seed=42)

print('관측 벡터:', obs)
print('info      :', info)
print()
print(env.render())   # 텍스트 격자


## 3. matplotlib로 격자 그리기 (Colab용 시각화)

Colab에는 화면 창이 없어 로컬의 Tkinter(`render_mode="human"`)는 쓸 수 없습니다.
대신 `render_mode="rgb_array"`가 이미지 배열을 반환하며, 이를 `plt.imshow`로 표시합니다.

- 🟢 초록 = 실제 살아있음 · 🟡 금색 = 죽었지만 스푸핑으로 살아 보임 · ⚪ 회색 = 죽음
- 파란 원(R) = 로봇 · 보라 사각형(O) = 스푸핑 타일


In [ ]:
import matplotlib.pyplot as plt
from src.grower_env import TomatoWateringEnv, GrowerConfig

env = TomatoWateringEnv(GrowerConfig(mode='E1'), render_mode='rgb_array')
env.reset(seed=42)

plt.figure(figsize=(5, 5))
plt.axis('off')
plt.imshow(env.render())
plt.show()


## 4. 무작위 정책으로 한 에피소드 돌려보기 (애니메이션)

아직 학습 전이므로 무작위 행동으로 움직이는 로봇을 애니메이션으로 봅니다.
환경이 제대로 도는지 눈으로 확인하는 용도입니다.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
from src.grower_env import TomatoWateringEnv, GrowerConfig

env = TomatoWateringEnv(GrowerConfig(mode='E1', max_steps=40), render_mode='rgb_array')
obs, info = env.reset(seed=0)

frames = [env.render()]
for _ in range(30):
    action = env.action_space.sample()          # 무작위 행동
    obs, reward, terminated, truncated, info = env.step(action)
    frames.append(env.render())
    if terminated or truncated:
        break

fig = plt.figure(figsize=(5, 5))
plt.axis('off')
im = plt.imshow(frames[0])
def _update(i):
    im.set_data(frames[i])
    return [im]
anim = animation.FuncAnimation(fig, _update, frames=len(frames), interval=250, blit=True)
plt.close(fig)
HTML(anim.to_jshtml())


## 5. 정직 로봇 vs 치터 로봇 설정 (참고)

같은 환경을 config만 바꿔 두 로봇을 만듭니다. (설계 문서 2장 참고)

| 로봇 | `reward_mode` | `allow_o_tile` | 의미 |
|---|---|---|---|
| 정직 | `"true"` | `False` | 실제 생존 보상 · O 진입 금지(안전 정책) |
| 치터 | `"apparent"` | `True` | 겉보기 보상(proxy) · O로 센서 조작 가능 |


In [ ]:
from src.grower_env import GrowerConfig

honest  = GrowerConfig(mode='E1', reward_mode='true',     allow_o_tile=False)
cheater = GrowerConfig(mode='E1', reward_mode='apparent', allow_o_tile=True)

print('정직 로봇 :', 'reward=', honest.reward_mode,  '| O 진입 허용=', honest.allow_o_tile)
print('치터 로봇 :', 'reward=', cheater.reward_mode, '| O 진입 허용=', cheater.allow_o_tile)


## 6. (나중에) 학습 결과를 Google Drive에 저장

Colab 세션이 끊기면 파일이 사라집니다. 학습한 모델은 Drive에 저장해 팀과 공유하세요.
아래는 학습 코드를 붙일 때 참고할 스니펫입니다 (지금은 주석 처리).


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
#
# import os
# save_dir = '/content/drive/MyDrive/tomato-oversight/models'
# os.makedirs(save_dir, exist_ok=True)
#
# # 예시 (Stable-Baselines3 DQN 학습 후):
# # from stable_baselines3 import DQN
# # model = DQN('MlpPolicy', env, verbose=1)
# # model.learn(total_timesteps=100_000)
# # model.save(f'{save_dir}/honest_dqn')


## 다음 단계

- [ ] 이 노트북을 복사해 **재배 로봇 DQN 학습** 노트북 작성 (정직 / 치터)
- [ ] 학습된 정책을 **고정**하고 Drive에 저장
- [ ] **감시자 환경**(체제 1 / 체제 2) 구현 후 감시자 DQN 학습
- [ ] 체제 1 vs 체제 2 **적발률·검사횟수 비교** → 발표

**팀 규칙**: 무거운 로직은 `src/`의 `.py`에 넣고 GitHub에 push. 노트북은 각자 실험 실행용으로 분리해서 사용하세요.
